In [1]:
import pandas as pd
import math

df = pd.read_csv("../../datasets/heart.csv")

numerical_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]
numerical_columns = [col for col in numerical_columns if col in df.columns]

print("Dataset shape:", df.shape)
print("Numerical columns:", numerical_columns)
display(df.head())

Dataset shape: (1025, 14)
Numerical columns: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [2]:
def manual_mean(values):
    values = [float(x) for x in values if pd.notna(x)]
    if len(values) == 0:
        return 0
    return sum(values) / len(values)


def manual_median(values):
    values = sorted(float(x) for x in values if pd.notna(x))
    n = len(values)

    if n == 0:
        return 0

    mid = n // 2

    if n % 2 == 1:
        return values[mid]

    return (values[mid - 1] + values[mid]) / 2


def manual_percentile(values, percentile):
    values = sorted(float(x) for x in values if pd.notna(x))

    if len(values) == 0:
        return 0

    position = (len(values) - 1) * percentile / 100
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    fraction = position - lower

    return values[lower] + fraction * (
        values[upper] - values[lower]
    )


def manual_std(values):
    values = [float(x) for x in values if pd.notna(x)]

    if len(values) < 2:
        return 0

    mean = manual_mean(values)
    variance = sum((x - mean) ** 2 for x in values) / len(values)

    return math.sqrt(variance)

In [3]:
df_clean = df.copy()

for col in numerical_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    values = df_clean[col].dropna().tolist()

    if len(values) == 0:
        continue

    q1 = manual_percentile(values, 25)
    q3 = manual_percentile(values, 75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_count = 0

    for value in values:
        if value < lower_limit or value > upper_limit:
            outlier_count += 1

    df_clean[col] = df_clean[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )

    print(f"\nColumn: {col}")
    print("Q1:", round(q1, 2))
    print("Q3:", round(q3, 2))
    print("Lower limit:", round(lower_limit, 2))
    print("Upper limit:", round(upper_limit, 2))
    print("Outliers clipped:", outlier_count)


Column: age
Q1: 48.0
Q3: 61.0
Lower limit: 28.5
Upper limit: 80.5
Outliers clipped: 0

Column: trestbps
Q1: 120.0
Q3: 140.0
Lower limit: 90.0
Upper limit: 170.0
Outliers clipped: 30

Column: chol
Q1: 211.0
Q3: 275.0
Lower limit: 115.0
Upper limit: 371.0
Outliers clipped: 16

Column: thalach
Q1: 132.0
Q3: 166.0
Lower limit: 81.0
Upper limit: 217.0
Outliers clipped: 4

Column: oldpeak
Q1: 0.0
Q3: 1.8
Lower limit: -2.7
Upper limit: 4.5
Outliers clipped: 7


In [4]:
df_normalized = df_clean.copy()
df_standardized = df_clean.copy()

for col in numerical_columns:
    values = df_clean[col].dropna().tolist()

    if len(values) == 0:
        continue

    minimum = min(values)
    maximum = max(values)

    mean = manual_mean(values)
    std = manual_std(values)

    normalized_values = []
    standardized_values = []

    for value in df_clean[col]:
        if pd.isna(value):
            normalized_values.append(float("nan"))
            standardized_values.append(float("nan"))
        else:
            if maximum != minimum:
                normalized_values.append(
                    (value - minimum) / (maximum - minimum)
                )
            else:
                normalized_values.append(0.0)

            if std != 0:
                standardized_values.append((value - mean) / std)
            else:
                standardized_values.append(0.0)

    df_normalized[col] = normalized_values
    df_standardized[col] = standardized_values

print("Min-Max Normalization:")
display(df_normalized[numerical_columns].head())

print("Z-score Standardization:")
display(df_standardized[numerical_columns].head())

Min-Max Normalization:


,age,trestbps,chol,thalach,oldpeak
0,0.479167,0.407895,0.351020,0.719008,0.222222
1,0.500000,0.605263,0.314286,0.611570,0.688889
2,0.854167,0.671053,0.195918,0.363636,0.577778
3,0.666667,0.710526,0.314286,0.661157,0.000000
4,0.687500,0.578947,0.685714,0.206612,0.422222


Z-score Standardization:


,age,trestbps,chol,thalach,oldpeak
0,-0.268437,-0.378869,-0.691104,0.824084,-0.054537
1,-0.158157,0.528894,-0.879693,0.255654,1.785457
2,1.716595,0.831481,-1.487368,-1.056105,1.347363
3,0.724079,1.013034,-0.879693,0.518006,-0.930725
4,0.834359,0.407859,1.027150,-1.886886,0.734031


In [5]:
df_transformed = df_clean.copy()

for col in numerical_columns:
    values = df_clean[col].dropna().tolist()

    if len(values) < 3:
        continue

    mean = manual_mean(values)
    std = manual_std(values)

    if std == 0:
        continue

    skewness = sum(
        ((x - mean) / std) ** 3 for x in values
    ) / len(values)

    print(f"{col} skewness: {skewness:.3f}")

    if abs(skewness) > 1:
        df_transformed[col] = df_clean[col].apply(
            lambda x: (
                math.copysign(math.log1p(abs(x)), x)
                if pd.notna(x) else float("nan")
            )
        )

        print(f"  -> Signed log transformation applied to {col}")
    else:
        print(f"  -> Transformation not required for {col}")

age skewness: -0.249
  -> Transformation not required for age
trestbps skewness: 0.402
  -> Transformation not required for trestbps
chol skewness: 0.343
  -> Transformation not required for chol
thalach skewness: -0.474
  -> Transformation not required for thalach
oldpeak skewness: 1.018
  -> Signed log transformation applied to oldpeak


In [6]:
df_clean.to_csv("../../datasets/heart_outlier_handled.csv", index=False)
df_normalized.to_csv("../../datasets/heart_normalized.csv", index=False)
df_standardized.to_csv("../../datasets/heart_standardized.csv", index=False)
df_transformed.to_csv("../../datasets/heart_transformed.csv", index=False)

print("All output files saved successfully.")

print("\nNormalized data range:")
for col in numerical_columns:
    values = df_normalized[col].dropna()

    if len(values) > 0:
        print(col, "Min:", round(values.min(), 3),
              "Max:", round(values.max(), 3))

print("\nStandardized data statistics:")
for col in numerical_columns:
    values = df_standardized[col].dropna()

    if len(values) > 0:
        print(
            col,
            "Mean:", round(manual_mean(values.tolist()), 3),
            "Std:", round(manual_std(values.tolist()), 3)
        )

All output files saved successfully.

Normalized data range:
age Min: 0.0 Max: 1.0
trestbps Min: 0.0 Max: 1.0
chol Min: 0.0 Max: 1.0
thalach Min: 0.0 Max: 1.0
oldpeak Min: 0.0 Max: 1.0

Standardized data statistics:
age Mean: -0.0 Std: 1.0
trestbps Mean: 0.0 Std: 1.0
chol Mean: 0.0 Std: 1.0
thalach Mean: -0.0 Std: 1.0
oldpeak Mean: -0.0 Std: 1.0


# Task 7: Manual Normalization and Standardization

## Objective
To normalize and standardize numerical features using manual calculations while handling outliers and skewed distributions.

## Methodology
1. Loaded the Heart Disease dataset using Pandas.
2. Implemented manual functions for mean, median, percentile, and standard deviation.
3. Detected outliers using the Interquartile Range (IQR) method and clipped extreme values.
4. Applied Min-Max normalization to scale features between 0 and 1.
5. Applied Z-score standardization using the mean and standard deviation.
6. Calculated skewness manually and applied a signed logarithmic transformation to highly skewed features.
7. Saved the processed datasets separately for comparison.

## Conclusion
Manual scaling and transformation methods were implemented without built-in scaling functions. Outlier clipping reduces the influence of extreme observations, normalization rescales feature ranges, standardization expresses values relative to their distribution, and logarithmic transformation can reduce skewness.

The generated outputs should be checked to confirm the transformations and their effects on each feature.
